# Campaign Performance Analysis

Analyze marketing campaigns across channels and launch dates to evaluate impressions, clicks, conversions, CPA, and ROI. The notebook emphasizes fair campaign comparison, channel-level tradeoffs, and the execution patterns that separate strong campaigns from weak ones.


## Project Overview

This project evaluates campaign efficiency using a public marketing workbook that includes campaign metadata, channel, launch date, impressions, clicks, conversion rate, acquisition cost, and ROI. The analysis combines descriptive KPIs, time trends, channel and campaign-type comparisons, and a composite effectiveness score so top campaigns are not judged on a single metric alone.


## Learning Objectives

- Build a reproducible campaign KPI layer from raw marketing records.
- Compare channels and campaign types on reach, efficiency, and return.
- Separate statistically noticeable differences from practically meaningful ones.
- Explain why top campaigns perform well using CTR, conversion rate, and CPA together.


## Business / Research Problem

Marketing teams rarely struggle to see spend and clicks. The harder question is whether campaigns are efficient after accounting for reach quality, conversion quality, and cost to acquire. This notebook asks four concrete questions:

1. Which channels and campaign types produce the best balance of reach and efficiency?
2. How stable are campaign KPIs over the year?
3. What separates top campaigns from bottom-quartile campaigns?
4. Are channel differences truly meaningful, or are winners mostly driven by campaign execution quality?


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy import stats

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 5)

DATA_URL = "https://raw.githubusercontent.com/Ayeni01/Market-Campaign-Analysis/main/marketing_campaign_dataset.xlsx"
LOCAL_DATA_PATH = Path("marketing_campaign_dataset.xlsx")

print("Imports and settings loaded.")


## Dataset Overview

The workbook contains one record per campaign with company, campaign type, target audience, channel, acquisition cost, ROI, clicks, impressions, engagement score, customer segment, and a launch date stored in Excel serial-date format. If a local workbook is placed beside the notebook it will be used; otherwise the notebook reads the public raw file directly from GitHub.


In [ ]:
data_source = LOCAL_DATA_PATH if LOCAL_DATA_PATH.exists() else DATA_URL
df_raw = pd.read_excel(data_source)

print(f"Data source: {data_source}")
print(f"Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
display(df_raw.head())
display(df_raw.sample(5, random_state=42))


## Data Validation Checks

Before engineering KPIs, validate record counts, duplicate campaign IDs, missing values, and whether Excel serial dates convert cleanly into calendar dates.


In [ ]:
parsed_dates = pd.to_datetime(df_raw["Date"], unit="D", origin="1899-12-30", errors="coerce")

validation_report = pd.DataFrame(
    {
        "metric": [
            "rows",
            "columns",
            "duplicate campaign ids",
            "missing parsed dates",
            "date min",
            "date max",
        ],
        "value": [
            len(df_raw),
            df_raw.shape[1],
            int(df_raw["Campaign_ID"].duplicated().sum()),
            int(parsed_dates.isna().sum()),
            parsed_dates.min(),
            parsed_dates.max(),
        ],
    }
)

missing_values = df_raw.isna().sum().rename("missing_values").to_frame()
numeric_summary = df_raw.describe(numeric_only=True).T[["mean", "std", "min", "max"]]

display(validation_report)
display(missing_values[missing_values["missing_values"] > 0].sort_values("missing_values", ascending=False))
display(numeric_summary)


## Data Cleaning And KPI Engineering

Campaigns already include ROI and conversion rate, but the notebook still needs derived metrics for direct comparison: conversions, CTR, CPC, CPA, CPM, calendar features, and a composite effectiveness score for fair ranking.


In [ ]:
df = df_raw.copy()
df.columns = [column.strip().lower() for column in df.columns]

numeric_cols = [
    "conversion_rate",
    "acquisition_cost",
    "roi",
    "clicks",
    "impressions",
    "engagement_score",
]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric)

df["date"] = pd.to_datetime(df["date"], unit="D", origin="1899-12-30")
df["duration_days"] = df["duration"].str.extract(r"(\d+)").astype(int)
df["conversions"] = (df["clicks"] * df["conversion_rate"]).round().astype(int)
df["ctr"] = df["clicks"] / df["impressions"]
df["cpc"] = df["acquisition_cost"] / df["clicks"]
df["cpa"] = df["acquisition_cost"] / df["conversions"]
df["cpm"] = 1000 * df["acquisition_cost"] / df["impressions"]
df["month"] = df["date"].dt.to_period("M").astype(str)
df["month_num"] = df["date"].dt.month
df["month_name"] = df["date"].dt.strftime("%b")
df["quarter"] = "Q" + df["date"].dt.quarter.astype(str)
df["weekday"] = df["date"].dt.day_name()

df["effectiveness_score"] = 100 * (
    0.35 * df["roi"].rank(pct=True)
    + 0.25 * df["conversion_rate"].rank(pct=True)
    + 0.20 * df["ctr"].rank(pct=True)
    + 0.10 * df["engagement_score"].rank(pct=True)
    + 0.10 * (1 - df["cpa"].rank(pct=True))
)

engineered_columns = [
    "campaign_id",
    "company",
    "campaign_type",
    "channel_used",
    "date",
    "clicks",
    "impressions",
    "conversions",
    "ctr",
    "conversion_rate",
    "cpa",
    "roi",
    "effectiveness_score",
]
display(df[engineered_columns].head())


## Exploratory Data Analysis

Start with a business-facing KPI snapshot before drilling into distributions and comparisons.


In [ ]:
kpi_summary = pd.DataFrame(
    {
        "metric": [
            "Campaigns",
            "Channels",
            "Campaign types",
            "Date range",
            "Total impressions",
            "Total clicks",
            "Estimated conversions",
            "Total acquisition cost",
            "Average CTR",
            "Average conversion rate",
            "Average CPA",
            "Average ROI",
        ],
        "value": [
            f"{len(df):,}",
            df["channel_used"].nunique(),
            df["campaign_type"].nunique(),
            f"{df['date'].min().date()} to {df['date'].max().date()}",
            f"{df['impressions'].sum():,}",
            f"{df['clicks'].sum():,}",
            f"{df['conversions'].sum():,}",
            f"${df['acquisition_cost'].sum():,.0f}",
            f"{100 * df['ctr'].mean():.2f}%",
            f"{100 * df['conversion_rate'].mean():.2f}%",
            f"${df['cpa'].mean():,.2f}",
            f"{df['roi'].mean():.2f}",
        ],
    }
)

display(kpi_summary)


## Univariate Analysis

The first pass checks the shape and spread of volume, cost, and return metrics. This is useful for spotting skew, outliers, and whether the dataset looks operationally noisy or tightly controlled.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

metric_order = [
    ("impressions", "Impressions"),
    ("clicks", "Clicks"),
    ("conversions", "Estimated Conversions"),
    ("acquisition_cost", "Acquisition Cost"),
    ("cpa", "CPA"),
    ("roi", "ROI"),
]

for axis, (column, title) in zip(axes.flat, metric_order):
    sns.histplot(df[column], bins=40, kde=True, ax=axis, color="#2a9d8f")
    axis.set_title(title)
    axis.set_xlabel(title)

plt.tight_layout()
plt.show()


## Bivariate / Multivariate Analysis

These plots focus on how efficiency metrics move together. Because the workbook contains more than 200,000 campaigns, the scatter plots use a reproducible sample for readability.


In [ ]:
sampled = df.sample(min(len(df), 12000), random_state=42)
spearman_corr = df[["roi", "ctr", "conversion_rate", "engagement_score", "cpa"]].corr(method="spearman")

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

sns.scatterplot(data=sampled, x="ctr", y="roi", hue="channel_used", alpha=0.35, ax=axes[0, 0], legend=False)
axes[0, 0].set_title("ROI vs CTR")

sns.scatterplot(data=sampled, x="conversion_rate", y="cpa", hue="campaign_type", alpha=0.35, ax=axes[0, 1], legend=False)
axes[0, 1].set_title("CPA vs Conversion Rate")

sns.scatterplot(data=sampled, x="engagement_score", y="ctr", hue="channel_used", alpha=0.35, ax=axes[1, 0], legend=False)
axes[1, 0].set_title("CTR vs Engagement Score")

sns.heatmap(spearman_corr, annot=True, fmt=".2f", cmap="coolwarm", ax=axes[1, 1])
axes[1, 1].set_title("Spearman Correlation Matrix")

plt.tight_layout()
plt.show()
display(spearman_corr.round(3))


## Channel Performance

Channel-level aggregation answers whether performance differences come from media choice or whether channels look broadly interchangeable once campaigns are scaled up.


In [ ]:
channel_summary = (
    df.groupby("channel_used")
    .agg(
        campaigns=("campaign_id", "count"),
        spend=("acquisition_cost", "sum"),
        impressions=("impressions", "sum"),
        clicks=("clicks", "sum"),
        conversions=("conversions", "sum"),
        avg_roi=("roi", "mean"),
        avg_engagement=("engagement_score", "mean"),
    )
    .reset_index()
)

channel_summary["ctr"] = channel_summary["clicks"] / channel_summary["impressions"]
channel_summary["conversion_rate"] = channel_summary["conversions"] / channel_summary["clicks"]
channel_summary["cpa"] = channel_summary["spend"] / channel_summary["conversions"]

display(channel_summary.sort_values("avg_roi", ascending=False).round(3))

fig, axes = plt.subplots(1, 2, figsize=(17, 5))
sns.barplot(data=channel_summary.sort_values("avg_roi", ascending=False), x="channel_used", y="avg_roi", ax=axes[0], palette="crest")
axes[0].set_title("Average ROI by Channel")
axes[0].tick_params(axis="x", rotation=30)

sns.barplot(data=channel_summary.sort_values("cpa"), x="channel_used", y="cpa", ax=axes[1], palette="flare")
axes[1].set_title("Average CPA by Channel")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()


## Campaign Type Performance

Campaign type highlights whether the objective of the campaign itself is a stronger driver than the distribution channel.


In [ ]:
type_summary = (
    df.groupby("campaign_type")
    .agg(
        campaigns=("campaign_id", "count"),
        spend=("acquisition_cost", "sum"),
        impressions=("impressions", "sum"),
        clicks=("clicks", "sum"),
        conversions=("conversions", "sum"),
        avg_roi=("roi", "mean"),
        avg_engagement=("engagement_score", "mean"),
    )
    .reset_index()
)

type_summary["ctr"] = type_summary["clicks"] / type_summary["impressions"]
type_summary["conversion_rate"] = type_summary["conversions"] / type_summary["clicks"]
type_summary["cpa"] = type_summary["spend"] / type_summary["conversions"]

display(type_summary.sort_values("avg_roi", ascending=False).round(3))

fig, axes = plt.subplots(1, 2, figsize=(17, 5))
sns.barplot(data=type_summary.sort_values("avg_roi", ascending=False), x="campaign_type", y="avg_roi", ax=axes[0], palette="viridis")
axes[0].set_title("Average ROI by Campaign Type")
axes[0].tick_params(axis="x", rotation=25)

sns.barplot(data=type_summary.sort_values("ctr", ascending=False), x="campaign_type", y="ctr", ax=axes[1], palette="mako")
axes[1].set_title("Average CTR by Campaign Type")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()


## Date And Trend Analysis

Time patterns matter because campaigns can look strong simply by launching in favorable weeks or months. This section checks whether performance materially changes across the year.


In [ ]:
monthly_summary = (
    df.groupby(["month_num", "month_name"])
    .agg(
        spend=("acquisition_cost", "sum"),
        clicks=("clicks", "sum"),
        conversions=("conversions", "sum"),
        avg_ctr=("ctr", "mean"),
        avg_conversion_rate=("conversion_rate", "mean"),
        avg_cpa=("cpa", "mean"),
        avg_roi=("roi", "mean"),
    )
    .reset_index()
    .sort_values("month_num")
)

display(monthly_summary.round(3))

fig, axes = plt.subplots(3, 1, figsize=(16, 14))

sns.lineplot(data=monthly_summary, x="month_name", y="clicks", marker="o", ax=axes[0], label="Clicks")
sns.lineplot(data=monthly_summary, x="month_name", y="conversions", marker="o", ax=axes[0], label="Conversions")
axes[0].set_title("Monthly Click And Conversion Volume")

sns.lineplot(data=monthly_summary, x="month_name", y="avg_ctr", marker="o", ax=axes[1], label="CTR")
sns.lineplot(data=monthly_summary, x="month_name", y="avg_conversion_rate", marker="o", ax=axes[1], label="Conversion Rate")
axes[1].set_title("Monthly Efficiency Rates")

sns.lineplot(data=monthly_summary, x="month_name", y="avg_cpa", marker="o", ax=axes[2], label="CPA")
sns.lineplot(data=monthly_summary, x="month_name", y="avg_roi", marker="o", ax=axes[2], label="ROI")
axes[2].set_title("Monthly Cost And Return")

for axis in axes:
    axis.grid(alpha=0.3)

plt.tight_layout()
plt.show()


## Feature-Specific Insights

Segment analysis checks whether the data suggests specific customer groups or audience clusters respond better, or whether campaign quality overwhelms segment differences.


In [ ]:
segment_summary = (
    df.groupby("customer_segment")
    .agg(
        campaigns=("campaign_id", "count"),
        avg_roi=("roi", "mean"),
        avg_ctr=("ctr", "mean"),
        avg_conversion_rate=("conversion_rate", "mean"),
        avg_cpa=("cpa", "mean"),
    )
    .reset_index()
    .sort_values("avg_roi", ascending=False)
)

display(segment_summary.round(3))

fig, axes = plt.subplots(1, 2, figsize=(17, 5))
sns.barplot(data=segment_summary.sort_values("avg_conversion_rate", ascending=False), x="customer_segment", y="avg_conversion_rate", ax=axes[0], palette="cubehelix")
axes[0].set_title("Average Conversion Rate By Segment")
axes[0].tick_params(axis="x", rotation=25)

sns.barplot(data=segment_summary.sort_values("avg_cpa"), x="customer_segment", y="avg_cpa", ax=axes[1], palette="rocket")
axes[1].set_title("Average CPA By Segment")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()


## Fair Campaign Comparison Logic

Ranking campaigns on raw ROI alone is weak here because ROI is very tightly distributed. A fairer scoreboard uses a median-activity filter plus a weighted effectiveness score built from ROI, conversion rate, CTR, engagement score, and inverse CPA.


In [ ]:
fair_pool = df[(df["impressions"] >= df["impressions"].median()) & (df["clicks"] >= df["clicks"].median())].copy()

leaderboard_columns = [
    "campaign_id",
    "company",
    "channel_used",
    "campaign_type",
    "date",
    "impressions",
    "clicks",
    "conversions",
    "ctr",
    "conversion_rate",
    "cpa",
    "roi",
    "effectiveness_score",
]

display(
    fair_pool.nlargest(15, "effectiveness_score")[leaderboard_columns].round(
        {
            "ctr": 4,
            "conversion_rate": 4,
            "cpa": 2,
            "roi": 2,
            "effectiveness_score": 2,
        }
    )
)

top_quartile = df[df["effectiveness_score"] >= df["effectiveness_score"].quantile(0.75)]
bottom_quartile = df[df["effectiveness_score"] <= df["effectiveness_score"].quantile(0.25)]

driver_profile = pd.DataFrame(
    {
        "top_quartile": [
            top_quartile["roi"].mean(),
            top_quartile["ctr"].mean(),
            top_quartile["conversion_rate"].mean(),
            top_quartile["cpa"].mean(),
            top_quartile["engagement_score"].mean(),
            top_quartile["duration_days"].mean(),
            top_quartile["clicks"].mean(),
            top_quartile["impressions"].mean(),
        ],
        "bottom_quartile": [
            bottom_quartile["roi"].mean(),
            bottom_quartile["ctr"].mean(),
            bottom_quartile["conversion_rate"].mean(),
            bottom_quartile["cpa"].mean(),
            bottom_quartile["engagement_score"].mean(),
            bottom_quartile["duration_days"].mean(),
            bottom_quartile["clicks"].mean(),
            bottom_quartile["impressions"].mean(),
        ],
    },
    index=[
        "roi",
        "ctr",
        "conversion_rate",
        "cpa",
        "engagement_score",
        "duration_days",
        "clicks",
        "impressions",
    ],
)

top_channel_share = (100 * top_quartile["channel_used"].value_counts(normalize=True)).round(2).rename("top_quartile_share_pct")
top_type_share = (100 * top_quartile["campaign_type"].value_counts(normalize=True)).round(2).rename("top_quartile_share_pct")

display(driver_profile.round(3))
display(top_channel_share.to_frame())
display(top_type_share.to_frame())


## Statistical Checks

Large marketing tables can make tiny differences look important. This section explicitly tests whether average ROI differs across channels or campaign types and reports effect size, not only p-values.


In [ ]:
channel_groups = [group["roi"].values for _, group in df.groupby("channel_used")]
type_groups = [group["roi"].values for _, group in df.groupby("campaign_type")]

channel_h, channel_p = stats.kruskal(*channel_groups)
type_h, type_p = stats.kruskal(*type_groups)

channel_k = len(channel_groups)
type_k = len(type_groups)
n = len(df)

channel_epsilon_sq = (channel_h - channel_k + 1) / (n - channel_k)
type_epsilon_sq = (type_h - type_k + 1) / (n - type_k)

stats_summary = pd.DataFrame(
    {
        "comparison": ["ROI by channel", "ROI by campaign type"],
        "kruskal_h": [channel_h, type_h],
        "p_value": [channel_p, type_p],
        "epsilon_squared": [channel_epsilon_sq, type_epsilon_sq],
    }
)

display(stats_summary.round(6))
display(df[["roi", "ctr", "conversion_rate", "engagement_score", "cpa"]].corr(method="spearman").round(3))


## Visual Analysis

The final visuals summarize whether channel performance changes materially month to month and whether the best-looking channels combine strong CTR and ROI without paying for it through high CPA.


In [ ]:
month_order = monthly_summary["month_name"].tolist()
roi_heatmap = df.pivot_table(index="channel_used", columns="month_name", values="roi", aggfunc="mean")
roi_heatmap = roi_heatmap[month_order]

bubble = channel_summary.copy()
bubble_sizes = 300 + 2200 * bubble["conversions"] / bubble["conversions"].max()

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.heatmap(roi_heatmap, annot=True, fmt=".2f", cmap="RdYlGn", ax=axes[0])
axes[0].set_title("Average ROI By Channel And Month")

scatter = axes[1].scatter(
    bubble["ctr"] * 100,
    bubble["avg_roi"],
    s=bubble_sizes,
    c=bubble["cpa"],
    cmap="viridis",
    alpha=0.8,
    edgecolor="black",
)
for _, row in bubble.iterrows():
    axes[1].text(row["ctr"] * 100, row["avg_roi"], row["channel_used"], ha="center", va="center", fontsize=10)

axes[1].set_xlabel("CTR (%)")
axes[1].set_ylabel("Average ROI")
axes[1].set_title("Channel Tradeoff: CTR, ROI, CPA, And Conversion Volume")
plt.colorbar(scatter, ax=axes[1], label="Average CPA")

plt.tight_layout()
plt.show()


## Key Findings

- Channel ROI is almost flat. Facebook is the top mean-ROI channel at about `5.02`, but the spread across channels is tiny and not statistically meaningful in a practical sense.
- Website and Email are the cheapest scaled channels on CPA, while Google Ads is the most expensive. The absolute gap is still modest relative to the dataset size.
- Top-quartile campaigns win through execution quality, not longer duration. They deliver roughly `20.2%` CTR versus `8.7%` for the bottom quartile, `11.0%` versus `4.9%` conversion rate, and about `$214` CPA versus `$1,295`.
- Bottom-quartile campaigns often buy more impressions but convert that attention poorly. Strong campaigns generate more clicks from fewer impressions, which implies better targeting or creative relevance.
- Channel and campaign-type shares in the top quartile are nearly even, so no single channel monopolizes top performers. The data points to operational quality inside campaigns rather than a simple "pick this channel" answer.
- Monthly trends are stable throughout 2021. The dataset does not show a strong seasonal spike that would explain winners by timing alone.


## Limitations

- The workbook is read from a public raw GitHub URL unless a local copy is provided, so notebook execution depends on network availability.
- Conversions are inferred from `Clicks * Conversion_Rate`; they are not stored as a native column.
- `Acquisition_Cost` is treated as campaign spend. If the source defines it differently, CPC and CPA interpretations should be revisited.
- The metric structure looks highly standardized, which suggests the dataset may be synthetic or heavily normalized. That likely compresses real-world channel differences.


## Recommendations / Next Steps

- Prioritize campaigns that lift CTR and conversion rate together instead of optimizing reach alone.
- Use channel as a secondary optimization lever; the strongest improvements are more likely to come from targeting, offer quality, landing-page quality, and creative fit.
- Add guardrails that flag campaigns with high impressions but weak click quality or very high CPA.
- Track top-quartile design patterns by company, audience, and creative, then reuse those playbooks in future launches.
- If richer data becomes available, extend the notebook with native revenue, post-click funnel steps, and A/B test metadata.


## Common Mistakes

- Ranking campaigns on ROI alone when ROI barely varies across the table.
- Treating statistical significance as business significance in a very large sample.
- Assuming more impressions means better performance without checking CTR and CPA.
- Blaming weak campaigns on channel choice when the main gap comes from execution quality.


## Mini Challenge

Build a second leaderboard that weights ROI, CPA, and conversion rate differently for three stakeholder views: a growth lead, a finance lead, and a brand lead. Compare how the top 10 campaigns change under each weighting scheme.


## Final Summary

The workbook shows that campaign winners are not mainly explained by channel or calendar timing. The best campaigns combine stronger click quality, better post-click conversion, and sharply lower CPA. In short: top campaigns are effective because they turn attention into action more efficiently, not because they simply bought more reach.
